# Grad-CAM Saliency Analysis — Deep Models

Qualitative explainability analysis for the three deep models using Grad-CAM, applied to the final v2 checkpoints and the per-clip predictions saved by the final test-evaluation notebook (`outputs_all_3_models_v2/final_test_eval/`).

Grad-CAM heat-maps highlight which facial regions drive each model's drowsiness decision, helping diagnose failure modes — in particular why models misclassify natural driving behaviours such as talking as drowsy (Chapter 5, Appendix A.4).

**Scenarios:**
1. Diagnostic case selection — confident TP / FP / hardest FN / cleanest TN
2. R(2+1)D-18 with layer3 (sharper maps)
3. YawDD talking false-positive
4. YawDD yawning true-positive

All cells use the locked thresholds (τ = 0.010 / 0.001 / 0.004) and read the saved prediction CSVs, so only the selected clips are re-run (fast).

---
## 1. Diagnostic Case Selection (TP / FP / FN / TN)

Reads saved predictions, selects the most confident true positive, most confident false positive, hardest false negative and cleanest true negative per split, and runs Grad-CAM only on those clips. Set `TARGET_SPLIT` to `UTA_test`, `YawDD_test` or `DROZY_test`.

In [ ]:
# =========================================================
# FAST GRAD-CAM USING SAVED PREDICTION CSVs
# - reads saved predictions from final_test_eval
# - selects:
#     * most confident TP
#     * most confident FP
#     * hardest FN
#     * cleanest TN
# - runs Grad-CAM only on those selected clips
# - saves PNG + JSON to Drive
#
# CHANGE ONLY:
#   TARGET_SPLIT = "UTA_test"   # or "YawDD_test" or "DROZY_test"
# =========================================================

# -------------------------
# 0) MOUNT DRIVE
# -------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# -------------------------
# 1) IMPORTS
# -------------------------
import os
import re
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision

from pathlib import Path
from torch.utils.data import Dataset
from tqdm import tqdm

# -------------------------
# 2) BASIC SETUP
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 2

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SPLITS_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/splits"
CLIPS_DRIVE  = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL  = "/content/Clips"

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
PRED_ROOT = os.path.join(SAVE_ROOT, "final_test_eval")
OUT_SUBDIR = "gradcam_diagnostic_from_preds"

# choose which prediction set to use
TARGET_SPLIT = "UTA_test"   # "UTA_test", "YawDD_test", or "DROZY_test"

# locked thresholds
THR = {
    "tsm_fast": 0.010,
    "cnn_gru":  0.001,
    "r3d18":    0.004
}

os.makedirs(SAVE_ROOT, exist_ok=True)

# -------------------------
# 3) LOAD THE MATCHING DATAFRAME
# -------------------------
def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = pd.Series([default_ds] * len(df), index=df.index)
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

if TARGET_SPLIT == "UTA_test":
    df_raw = namespace_subjects(load_split_csv(os.path.join(SPLITS_DIR, "uta_test.csv")), "uta")
elif TARGET_SPLIT == "YawDD_test":
    # if you have a yawdd_test.csv, use that here; otherwise this will fail
    df_raw = namespace_subjects(load_split_csv(os.path.join(SPLITS_DIR, "yawdd_test.csv")), "yawdd")
elif TARGET_SPLIT == "DROZY_test":
    df_raw = namespace_subjects(load_split_csv(os.path.join(SPLITS_DIR, "drozy_test.csv")), "drozy")
else:
    raise ValueError("TARGET_SPLIT must be one of: UTA_test, YawDD_test, DROZY_test")

CLIPS_DRIVE_P = Path(CLIPS_DRIVE)
CLIPS_LOCAL_P = Path(CLIPS_LOCAL)

def map_to_local_df(df):
    df = df.copy()
    def _map(p):
        p = str(p)
        pref = str(CLIPS_DRIVE_P) + "/"
        if p.startswith(pref):
            rel = p[len(pref):]
            local = CLIPS_LOCAL_P / rel
            if local.exists():
                return str(local)
        return p
    df["clip_path_mapped"] = df["clip_path"].astype(str).apply(_map)
    return df

df_m = map_to_local_df(df_raw)

print("Loaded split:", TARGET_SPLIT, "| n =", len(df_m))
print("Example path:", df_m["clip_path_mapped"].iloc[0])

# -------------------------
# 4) MODEL BUILDERS
# -------------------------
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous().view(B * T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, dropout=0.30, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        out_last = self.drop(out[:, -1, :])
        return self.fc(out_last)

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, dropout=0.30, num_classes=num_classes)

def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# -------------------------
# 5) ROBUST DATASET
# -------------------------
class NPZClipsDataset(Dataset):
    def __init__(self, df, mode="cthw", clip_t=16, clip_h=112, clip_w=112, clip_c=3):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.clip_t = clip_t
        self.clip_h = clip_h
        self.clip_w = clip_w
        self.clip_c = clip_c

    def __len__(self):
        return len(self.df)

    def _dummy_cthw(self):
        return np.zeros((self.clip_c, self.clip_t, self.clip_h, self.clip_w), dtype=np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["clip_path_mapped"]
        y = int(row["label"])

        bad = 0
        try:
            z = np.load(path, allow_pickle=False)
            x = z["x"] if "x" in z.files else z[z.files[0]]
            x = x.astype(np.float32)

            if x.ndim != 4:
                raise ValueError(f"bad ndim={x.ndim}")

            if x.shape[-1] in (1, 3):          # (T,H,W,C)
                x = np.transpose(x, (3, 0, 1, 2))
            elif x.shape[1] in (1, 3):         # (T,C,H,W)
                x = np.transpose(x, (1, 0, 2, 3))
            elif x.shape[0] in (1, 3):         # already (C,T,H,W)
                pass
            else:
                raise ValueError(f"unrecognized layout shape={x.shape}")

            expected = (self.clip_c, self.clip_t, self.clip_h, self.clip_w)
            if x.shape != expected:
                raise ValueError(f"wrong shape {x.shape} expected {expected}")

            if not np.isfinite(x).all():
                raise ValueError("nan/inf found")

        except Exception:
            bad = 1
            x = self._dummy_cthw()

        if self.mode == "cthw":
            xt = torch.from_numpy(x)
        elif self.mode == "tchw":
            xt = torch.from_numpy(np.transpose(x, (1, 0, 2, 3)))
        else:
            raise ValueError("mode must be cthw or tchw")

        return xt, torch.tensor(y, dtype=torch.long), torch.tensor(bad, dtype=torch.bool), path

# -------------------------
# 6) HELPERS
# -------------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def load_ckpt(builder, ckpt_path):
    model = builder(NUM_CLASSES).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()
    return model

def to_numpy_img(x_chw):
    x = x_chw.astype(np.float32)
    if x.max() > 1.5:
        x = x / 255.0
    x = np.clip(x, 0, 1)
    return np.transpose(x, (1, 2, 0))

def overlay_heatmap(img_hwc, cam_hw, alpha=0.45):
    cam = np.clip(cam_hw, 0, 1)
    heat = plt.get_cmap("jet")(cam)[:, :, :3]
    out = (1 - alpha) * img_hwc + alpha * heat
    return np.clip(out, 0, 1)

def save_grid(frames_hwc, cams_hw, title, save_path, max_frames=8):
    T = len(frames_hwc)
    if T <= max_frames:
        idxs = list(range(T))
    else:
        idxs = np.linspace(0, T - 1, max_frames).round().astype(int).tolist()

    n = len(idxs)
    cols = min(n, 4)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, t in enumerate(idxs):
        over = overlay_heatmap(frames_hwc[t], cams_hw[t])
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(over)
        ax.set_title(f"t={t}")
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def load_clip_by_path(df_m, clip_path, mode):
    hit = df_m.index[df_m["clip_path"].astype(str) == str(clip_path)]
    if len(hit) == 0:
        # fallback: filename match
        target_name = os.path.basename(str(clip_path))
        hit = df_m.index[df_m["clip_path"].astype(str).apply(lambda s: os.path.basename(s) == target_name)]
        if len(hit) == 0:
            return None

    idx = int(hit[0])
    ds = NPZClipsDataset(df_m, mode=mode)
    x, y, bad, path = ds[idx]
    if bool(bad.item()):
        return None
    return idx, x.clone(), int(y.item()), path

# -------------------------
# 7) READ PREDICTION CSVs
# -------------------------
def find_pred_file(model_name, split_name):
    fname = f"preds_{model_name}_{split_name}.csv"
    path = os.path.join(PRED_ROOT, fname)
    if os.path.exists(path):
        return path

    # fallback: search loosely
    cands = []
    for f in os.listdir(PRED_ROOT):
        ff = f.lower()
        if f.endswith(".csv") and model_name.lower() in ff and split_name.lower().replace("_", "")[:3] in ff:
            cands.append(os.path.join(PRED_ROOT, f))
    if len(cands) == 1:
        return cands[0]
    if len(cands) > 1:
        print(f"[WARN] multiple matches for {model_name}, {split_name}:")
        for c in cands:
            print("  ", c)
        return cands[0]

    raise FileNotFoundError(f"Could not find prediction CSV for {model_name} / {split_name} in {PRED_ROOT}")

def pick_col(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Could not find any of these columns: {candidates}\nAvailable: {list(df.columns)}")
    return None

def normalize_pred_df(pred_df):
    pred_df = pred_df.copy()

    path_col = pick_col(pred_df, [
        "clip_path", "path", "file_path", "npz_path", "clip"
    ])
    y_col = pick_col(pred_df, [
        "label", "y_true", "target", "gt", "true_label"
    ])
    p1_col = pick_col(pred_df, [
        "p1", "prob", "prob_1", "prob_drowsy", "p_drowsy",
        "score", "y_score", "pred_prob", "positive_prob", "class1_prob"
    ])

    out = pd.DataFrame({
        "clip_path": pred_df[path_col].astype(str),
        "y": pd.to_numeric(pred_df[y_col], errors="coerce"),
        "p1": pd.to_numeric(pred_df[p1_col], errors="coerce"),
    })

    out = out.dropna(subset=["clip_path", "y", "p1"]).copy()
    out["y"] = out["y"].astype(int)
    out["p1"] = out["p1"].astype(float)
    return out

def select_cases_from_preds(pred_df, thr):
    pred_df = pred_df.copy()
    pred_df["yhat"] = (pred_df["p1"] >= thr).astype(int)

    tp = pred_df[(pred_df["y"] == 1) & (pred_df["yhat"] == 1)].sort_values("p1", ascending=False)
    fp = pred_df[(pred_df["y"] == 0) & (pred_df["yhat"] == 1)].sort_values("p1", ascending=False)
    fn = pred_df[(pred_df["y"] == 1) & (pred_df["yhat"] == 0)].sort_values("p1", ascending=False)
    tn = pred_df[(pred_df["y"] == 0) & (pred_df["yhat"] == 0)].sort_values("p1", ascending=True)

    best = {
        "TP": None if len(tp) == 0 else tp.iloc[0].to_dict(),
        "FP": None if len(fp) == 0 else fp.iloc[0].to_dict(),
        "FN": None if len(fn) == 0 else fn.iloc[0].to_dict(),
        "TN": None if len(tn) == 0 else tn.iloc[0].to_dict(),
    }
    return best

# -------------------------
# 8) MODE HELPERS
# -------------------------
def set_bn_eval(m):
    if isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
        m.eval()

def set_dropout_eval(m):
    if isinstance(m, torch.nn.Dropout):
        m.eval()

# -------------------------
# 9) GRAD-CAM FOR TSM / CNN-GRU
# -------------------------
def gradcam_resnet18_video(model, x_cthw_or_tchw, mode, target_class=1, needs_rnn_train=False):
    conv = model.backbone[7][-1].conv2

    acts, grads = [], []

    def fwd_hook(m, inp, out):
        acts.append(out)

    def bwd_hook(m, gin, gout):
        grads.append(gout[0])

    h1 = conv.register_forward_hook(fwd_hook)
    h2 = conv.register_full_backward_hook(bwd_hook)

    was_training = model.training

    if needs_rnn_train:
        model.train()
        model.apply(set_bn_eval)
        model.apply(set_dropout_eval)
    else:
        model.eval()

    model.zero_grad(set_to_none=True)

    xb = x_cthw_or_tchw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    if was_training:
        model.train()
    else:
        model.eval()

    h1.remove()
    h2.remove()

    A = acts[0]
    G = grads[0]

    w = G.mean(dim=(2, 3), keepdim=True)
    cam = (w * A).sum(dim=1)
    cam = F.relu(cam)

    if mode == "cthw":
        T = xb.shape[2]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    elif mode == "tchw":
        T = xb.shape[1]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].detach().cpu().numpy()
    else:
        raise ValueError("mode must be 'cthw' or 'tchw'")

    cam = cam.view(T, cam.shape[-2], cam.shape[-1]).unsqueeze(1)
    cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
    cam = cam.squeeze(1)
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

# -------------------------
# 10) GRAD-CAM FOR R3D18
# -------------------------
def gradcam_r3d18(model, x_cthw, target_class=1, layer="layer4"):
    acts, grads = [], []

    if layer == "layer4":
        conv = model.layer4[1].conv2
    elif layer == "layer3":
        conv = model.layer3[1].conv2
    else:
        raise ValueError("layer must be 'layer3' or 'layer4'")

    def fwd_hook(m, inp, out):
        acts.append(out)
        out.register_hook(lambda g: grads.append(g))

    h = conv.register_forward_hook(fwd_hook)

    model.eval()
    model.zero_grad(set_to_none=True)

    xb = x_cthw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    h.remove()

    A = acts[0]
    G = grads[0]

    w = G.mean(dim=(2, 3, 4), keepdim=True)
    cam = (w * A).sum(dim=1)
    cam = F.relu(cam)

    T, H, W = xb.shape[2], xb.shape[3], xb.shape[4]
    cam = cam.unsqueeze(1)
    cam = F.interpolate(cam, size=(T, H, W), mode="trilinear", align_corners=False)
    cam = cam.squeeze(1)[0]
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

# -------------------------
# 11) RUN ONE MODEL
# -------------------------
def run_gradcam_for_model(model_name, builder, mode, thr, r3d_layer="layer4"):
    pred_path = find_pred_file(model_name, TARGET_SPLIT)
    pred_df_raw = pd.read_csv(pred_path)
    pred_df = normalize_pred_df(pred_df_raw)

    print("\n" + "=" * 70)
    print(f"Running {model_name} on {TARGET_SPLIT}")
    print("=" * 70)
    print("Prediction CSV:", pred_path)

    cases = select_cases_from_preds(pred_df, thr=thr)

    print("Selected cases from predictions:")
    for tag in ["TP", "FP", "FN", "TN"]:
        info = cases[tag]
        if info is None:
            print(f"  {tag}: not found")
        else:
            print(f"  {tag}: y={int(info['y'])} | p1={float(info['p1']):.6f} | path={info['clip_path']}")

    exp_dir = os.path.join(SAVE_ROOT, model_name)
    ckpt = os.path.join(exp_dir, "best.pt")
    assert os.path.exists(ckpt), f"Missing checkpoint: {ckpt}"

    out_dir = os.path.join(exp_dir, OUT_SUBDIR, TARGET_SPLIT)
    ensure_dir(out_dir)

    model = load_ckpt(builder, ckpt)

    for tag in tqdm(["TP", "FP", "FN", "TN"], desc=f"Grad-CAM {model_name}", leave=True):
        info = cases[tag]
        if info is None:
            continue

        loaded = load_clip_by_path(df_m, info["clip_path"], mode)
        if loaded is None:
            print(f"[WARN] Could not reload clip for {tag}: {info['clip_path']}")
            continue

        idx, x, y, path = loaded

        if model_name == "r3d18":
            frames_hwc, cams = gradcam_r3d18(model, x, target_class=1, layer=r3d_layer)
            title = f"{model_name}({r3d_layer}) | {TARGET_SPLIT} | {tag} | y={y} p1={float(info['p1']):.3f} thr={thr:.3f}"
            save_name = f"{model_name}_{r3d_layer}_{TARGET_SPLIT}_{tag.lower()}_idx{idx}.png"
        else:
            needs_rnn = (model_name == "cnn_gru")
            frames_hwc, cams = gradcam_resnet18_video(
                model, x, mode=mode, target_class=1, needs_rnn_train=needs_rnn
            )
            title = f"{model_name} | {TARGET_SPLIT} | {tag} | y={y} p1={float(info['p1']):.3f} thr={thr:.3f}"
            save_name = f"{model_name}_{TARGET_SPLIT}_{tag.lower()}_idx{idx}.png"

        save_path = os.path.join(out_dir, save_name)
        save_grid(frames_hwc, cams, title=title, save_path=save_path, max_frames=8)

        meta = {
            "model": model_name,
            "split": TARGET_SPLIT,
            "tag": tag,
            "idx": int(idx),
            "y": int(y),
            "p1": float(info["p1"]),
            "thr": float(thr),
            "pred_csv": pred_path,
            "clip_path_from_csv": info["clip_path"],
            "reloaded_path": path,
            "png": save_path,
        }
        if model_name == "r3d18":
            meta["layer"] = r3d_layer

        with open(save_path.replace(".png", ".json"), "w") as f:
            json.dump(meta, f, indent=2)

        print(f"✅ saved {tag} -> {save_path}")

# -------------------------
# 12) RUN ALL 3
# -------------------------
run_gradcam_for_model(
    model_name="tsm_fast",
    builder=build_tsm_fast,
    mode="cthw",
    thr=THR["tsm_fast"]
)

run_gradcam_for_model(
    model_name="cnn_gru",
    builder=build_cnn_gru,
    mode="tchw",
    thr=THR["cnn_gru"]
)

run_gradcam_for_model(
    model_name="r3d18",
    builder=build_r3d18,
    mode="cthw",
    thr=THR["r3d18"],
    r3d_layer="layer4"
)

print("\n✅ DONE")
print("Saved to:")
print("/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/<model>/gradcam_diagnostic_from_preds/<split>/")

---
## 2. R(2+1)D-18 Grad-CAM (layer3)

R(2+1)D-18 saliency using layer3 activations, which give sharper spatial maps than the final convolutional block for this architecture.

In [ ]:
# =========================================================
# R3D18 GRAD-CAM USING LAYER3
# - uses saved prediction CSV
# - selects TP / FP / FN / TN from predictions
# - runs Grad-CAM with layer3 (sharper maps)
# - saves PNG + JSON
# =========================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision

from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset

# -------------------------
# BASIC SETUP
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 2

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SPLITS_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/splits"
CLIPS_DRIVE  = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL  = "/content/Clips"

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
PRED_ROOT = os.path.join(SAVE_ROOT, "final_test_eval")

MODEL_NAME  = "r3d18"
TARGET_SPLIT = "UTA_test"   # change if needed
THR = 0.004
R3D_LAYER = "layer3"

OUT_SUBDIR = f"gradcam_diagnostic_from_preds_{R3D_LAYER}"

# -------------------------
# LOAD SPLIT
# -------------------------
def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = pd.Series([default_ds] * len(df), index=df.index)
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

if TARGET_SPLIT == "UTA_test":
    df_raw = namespace_subjects(load_split_csv(os.path.join(SPLITS_DIR, "uta_test.csv")), "uta")
elif TARGET_SPLIT == "DROZY_test":
    df_raw = namespace_subjects(load_split_csv(os.path.join(SPLITS_DIR, "drozy_test.csv")), "drozy")
elif TARGET_SPLIT == "YawDD_test":
    df_raw = namespace_subjects(load_split_csv(os.path.join(SPLITS_DIR, "yawdd_test.csv")), "yawdd")
else:
    raise ValueError("TARGET_SPLIT must be UTA_test, DROZY_test, or YawDD_test")

CLIPS_DRIVE_P = Path(CLIPS_DRIVE)
CLIPS_LOCAL_P = Path(CLIPS_LOCAL)

def map_to_local_df(df):
    df = df.copy()

    def _map(p):
        p = str(p)
        pref = str(CLIPS_DRIVE_P) + "/"
        if p.startswith(pref):
            rel = p[len(pref):]
            local = CLIPS_LOCAL_P / rel
            if local.exists():
                return str(local)
        return p

    df["clip_path_mapped"] = df["clip_path"].astype(str).apply(_map)
    return df

df_m = map_to_local_df(df_raw)
print("Loaded split:", TARGET_SPLIT, "| n =", len(df_m))

# -------------------------
# MODEL
# -------------------------
def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_ckpt(builder, ckpt_path):
    model = builder(NUM_CLASSES).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()
    return model

# -------------------------
# DATASET
# -------------------------
class NPZClipsDataset(Dataset):
    def __init__(self, df, mode="cthw", clip_t=16, clip_h=112, clip_w=112, clip_c=3):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.clip_t = clip_t
        self.clip_h = clip_h
        self.clip_w = clip_w
        self.clip_c = clip_c

    def __len__(self):
        return len(self.df)

    def _dummy_cthw(self):
        return np.zeros((self.clip_c, self.clip_t, self.clip_h, self.clip_w), dtype=np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["clip_path_mapped"]
        y = int(row["label"])

        bad = 0
        try:
            z = np.load(path, allow_pickle=False)
            x = z["x"] if "x" in z.files else z[z.files[0]]
            x = x.astype(np.float32)

            if x.ndim != 4:
                raise ValueError("bad ndim")

            if x.shape[-1] in (1, 3):
                x = np.transpose(x, (3, 0, 1, 2))   # (T,H,W,C)->(C,T,H,W)
            elif x.shape[1] in (1, 3):
                x = np.transpose(x, (1, 0, 2, 3))   # (T,C,H,W)->(C,T,H,W)
            elif x.shape[0] in (1, 3):
                pass
            else:
                raise ValueError("layout")

            exp = (self.clip_c, self.clip_t, self.clip_h, self.clip_w)
            if x.shape != exp:
                raise ValueError(f"shape {x.shape} != {exp}")

            if not np.isfinite(x).all():
                raise ValueError("nan/inf")

        except Exception:
            bad = 1
            x = self._dummy_cthw()

        xt = torch.from_numpy(x)  # r3d18 uses (C,T,H,W)
        return xt, torch.tensor(y, dtype=torch.long), torch.tensor(bad, dtype=torch.bool), path

# -------------------------
# HELPERS
# -------------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def to_numpy_img(x_chw):
    x = x_chw.astype(np.float32)
    if x.max() > 1.5:
        x = x / 255.0
    x = np.clip(x, 0, 1)
    return np.transpose(x, (1, 2, 0))

def overlay_heatmap(img_hwc, cam_hw, alpha=0.45):
    cam = np.clip(cam_hw, 0, 1)
    heat = plt.get_cmap("jet")(cam)[:, :, :3]
    out = (1 - alpha) * img_hwc + alpha * heat
    return np.clip(out, 0, 1)

def save_grid(frames_hwc, cams_hw, title, save_path, max_frames=8):
    T = len(frames_hwc)
    if T <= max_frames:
        idxs = list(range(T))
    else:
        idxs = np.linspace(0, T - 1, max_frames).round().astype(int).tolist()

    n = len(idxs)
    cols = min(n, 4)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, t in enumerate(idxs):
        over = overlay_heatmap(frames_hwc[t], cams_hw[t])
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(over)
        ax.set_title(f"t={t}")
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# PREDICTION CSV HELPERS
# -------------------------
def find_pred_file(model_name, split_name):
    fname = f"preds_{model_name}_{split_name}.csv"
    path = os.path.join(PRED_ROOT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return path

def pick_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise KeyError(f"Missing columns {candidates}; available={list(df.columns)}")

def normalize_pred_df(pred_df):
    path_col = pick_col(pred_df, ["clip_path", "path", "file_path", "npz_path", "clip"])
    y_col    = pick_col(pred_df, ["label", "y_true", "target", "gt", "true_label"])
    p1_col   = pick_col(pred_df, ["p1", "prob", "prob_1", "prob_drowsy", "p_drowsy", "score", "y_score", "pred_prob"])

    out = pd.DataFrame({
        "clip_path": pred_df[path_col].astype(str),
        "y": pd.to_numeric(pred_df[y_col], errors="coerce"),
        "p1": pd.to_numeric(pred_df[p1_col], errors="coerce"),
    })
    out = out.dropna(subset=["clip_path", "y", "p1"]).copy()
    out["y"] = out["y"].astype(int)
    out["p1"] = out["p1"].astype(float)
    return out

def select_cases_from_preds(pred_df, thr):
    pred_df = pred_df.copy()
    pred_df["yhat"] = (pred_df["p1"] >= thr).astype(int)

    tp = pred_df[(pred_df["y"] == 1) & (pred_df["yhat"] == 1)].sort_values("p1", ascending=False)
    fp = pred_df[(pred_df["y"] == 0) & (pred_df["yhat"] == 1)].sort_values("p1", ascending=False)
    fn = pred_df[(pred_df["y"] == 1) & (pred_df["yhat"] == 0)].sort_values("p1", ascending=False)
    tn = pred_df[(pred_df["y"] == 0) & (pred_df["yhat"] == 0)].sort_values("p1", ascending=True)

    return {
        "TP": None if len(tp) == 0 else tp.iloc[0].to_dict(),
        "FP": None if len(fp) == 0 else fp.iloc[0].to_dict(),
        "FN": None if len(fn) == 0 else fn.iloc[0].to_dict(),
        "TN": None if len(tn) == 0 else tn.iloc[0].to_dict(),
    }

# -------------------------
# STRONG PATH MATCHER
# -------------------------
def _norm_path(s):
    s = str(s).replace("\\", "/").strip()
    s = s.replace("/content/drive/MyDrive/ES327_Drowsiness/Clips/", "")
    s = s.replace("/content/Clips/", "")
    return s

def _basename_noext(s):
    return os.path.splitext(os.path.basename(str(s)))[0]

def load_clip_by_path_strong(df_m, clip_path, mode="cthw"):
    clip_path = str(clip_path)
    target_norm = _norm_path(clip_path)
    target_base = os.path.basename(target_norm)
    target_stem = _basename_noext(target_norm)

    col_clip = df_m["clip_path"].astype(str)
    col_map  = df_m["clip_path_mapped"].astype(str)

    clip_norm = col_clip.apply(_norm_path)
    map_norm  = col_map.apply(_norm_path)

    hits = df_m.index[clip_norm == target_norm].tolist()
    if len(hits) == 0:
        hits = df_m.index[map_norm == target_norm].tolist()
    if len(hits) == 0:
        hits = df_m.index[col_clip.apply(lambda s: os.path.basename(str(s)) == target_base)].tolist()
    if len(hits) == 0:
        hits = df_m.index[col_map.apply(lambda s: os.path.basename(str(s)) == target_base)].tolist()
    if len(hits) == 0:
        hits = df_m.index[col_clip.apply(lambda s: _basename_noext(s) == target_stem)].tolist()
    if len(hits) == 0:
        hits = df_m.index[col_map.apply(lambda s: _basename_noext(s) == target_stem)].tolist()
    if len(hits) == 0:
        hits = df_m.index[clip_norm.apply(lambda s: s.endswith(target_base))].tolist()
    if len(hits) == 0:
        hits = df_m.index[map_norm.apply(lambda s: s.endswith(target_base))].tolist()

    if len(hits) == 0:
        print(f"[WARN] No match found for: {clip_path}")
        return None

    idx = int(hits[0])
    ds = NPZClipsDataset(df_m, mode=mode)
    x, y, bad, path = ds[idx]

    if bool(bad.item()):
        print(f"[WARN] Matched row but NPZ marked bad: {path}")
        return None

    return idx, x.clone(), int(y.item()), path

# -------------------------
# R3D18 GRAD-CAM USING LAYER3
# -------------------------
def gradcam_r3d18(model, x_cthw, target_class=1, layer="layer3"):
    acts, grads = [], []

    if layer == "layer4":
        conv = model.layer4[1].conv2
    elif layer == "layer3":
        conv = model.layer3[1].conv2
    else:
        raise ValueError("layer must be 'layer3' or 'layer4'")

    def fwd_hook(m, inp, out):
        acts.append(out)
        out.register_hook(lambda g: grads.append(g))

    h = conv.register_forward_hook(fwd_hook)

    model.eval()
    model.zero_grad(set_to_none=True)

    xb = x_cthw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    h.remove()

    A = acts[0]
    G = grads[0]

    w = G.mean(dim=(2, 3, 4), keepdim=True)
    cam = (w * A).sum(dim=1)
    cam = F.relu(cam)

    T, H, W = xb.shape[2], xb.shape[3], xb.shape[4]
    cam = cam.unsqueeze(1)
    cam = F.interpolate(cam, size=(T, H, W), mode="trilinear", align_corners=False)
    cam = cam.squeeze(1)[0]
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

# -------------------------
# RUN
# -------------------------
pred_path = find_pred_file(MODEL_NAME, TARGET_SPLIT)
pred_df_raw = pd.read_csv(pred_path)
pred_df = normalize_pred_df(pred_df_raw)
cases = select_cases_from_preds(pred_df, thr=THR)

print("\nSelected cases:")
for tag in ["TP", "FP", "FN", "TN"]:
    info = cases[tag]
    if info is None:
        print(f"  {tag}: not found")
    else:
        print(f"  {tag}: y={int(info['y'])} | p1={float(info['p1']):.6f} | path={info['clip_path']}")

exp_dir = os.path.join(SAVE_ROOT, MODEL_NAME)
ckpt = os.path.join(exp_dir, "best.pt")
assert os.path.exists(ckpt), f"Missing checkpoint: {ckpt}"

out_dir = os.path.join(exp_dir, OUT_SUBDIR, TARGET_SPLIT)
ensure_dir(out_dir)

model = load_ckpt(build_r3d18, ckpt)

for tag in tqdm(["TP", "FP", "FN", "TN"], desc=f"{MODEL_NAME} {R3D_LAYER}", leave=True):
    info = cases[tag]
    if info is None:
        continue

    loaded = load_clip_by_path_strong(df_m, info["clip_path"], mode="cthw")
    if loaded is None:
        print(f"[WARN] Could not reload clip for {tag}: {info['clip_path']}")
        continue

    idx, x, y, path = loaded

    frames_hwc, cams = gradcam_r3d18(model, x, target_class=1, layer=R3D_LAYER)

    title = f"{MODEL_NAME}({R3D_LAYER}) | {TARGET_SPLIT} | {tag} | y={y} p1={float(info['p1']):.3f} thr={THR:.3f}"
    save_name = f"{MODEL_NAME}_{R3D_LAYER}_{TARGET_SPLIT}_{tag.lower()}_idx{idx}.png"
    save_path = os.path.join(out_dir, save_name)

    save_grid(frames_hwc, cams, title=title, save_path=save_path, max_frames=8)

    meta = {
        "model": MODEL_NAME,
        "layer": R3D_LAYER,
        "split": TARGET_SPLIT,
        "tag": tag,
        "idx": int(idx),
        "y": int(y),
        "p1": float(info["p1"]),
        "thr": float(THR),
        "pred_csv": pred_path,
        "clip_path_from_csv": info["clip_path"],
        "reloaded_path": path,
        "png": save_path,
    }

    with open(save_path.replace(".png", ".json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"✅ saved {tag} -> {save_path}")

print("\n✅ Done")
print("Saved to:", out_dir)

---
## 3. YawDD Talking False-Positive

Selects a talking clip that the model misclassified as drowsy (false positive) and visualises where the model attends — central to the discussion of why talking triggers false alarms.

In [ ]:
# =========================================================
# YAWDD TALKING FALSE POSITIVE + GRAD-CAM (ONE CELL, FIXED)
# - falls back from /content/Clips/... to Drive path if needed
# =========================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision

# -------------------------
# USER SETTINGS
# -------------------------
MODEL_NAME = "cnn_gru"   # "cnn_gru", "tsm_fast", or "r3d18"
TARGET_SPLIT = "YawDD_test"

TALK_KEYWORDS = [
    "talk", "talking", "speak", "speaking", "speech",
    "conversation", "phone", "chat"
]

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
PRED_ROOT = os.path.join(SAVE_ROOT, "final_test_eval")
OUT_DIR   = os.path.join(SAVE_ROOT, MODEL_NAME, "gradcam_talking_fp", TARGET_SPLIT)

CLIPS_DRIVE = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL = "/content/Clips"

THR = {
    "tsm_fast": 0.010,
    "cnn_gru":  0.001,
    "r3d18":    0.004
}

# -------------------------
# SETUP
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 2
os.makedirs(OUT_DIR, exist_ok=True)

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

thr = THR[MODEL_NAME]

# -------------------------
# MODEL BUILDERS
# -------------------------
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous().view(B * T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, dropout=0.30, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        out_last = self.drop(out[:, -1, :])
        return self.fc(out_last)

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, dropout=0.30, num_classes=num_classes)

def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

BUILDERS = {
    "tsm_fast": build_tsm_fast,
    "cnn_gru": build_cnn_gru,
    "r3d18": build_r3d18
}

MODES = {
    "tsm_fast": "cthw",
    "cnn_gru": "tchw",
    "r3d18": "cthw"
}

def load_ckpt(builder, ckpt_path):
    model = builder(NUM_CLASSES).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()
    return model

ckpt_path = os.path.join(SAVE_ROOT, MODEL_NAME, "best.pt")
assert os.path.exists(ckpt_path), f"Missing checkpoint: {ckpt_path}"
model = load_ckpt(BUILDERS[MODEL_NAME], ckpt_path)
mode = MODES[MODEL_NAME]

# -------------------------
# PREDICTION CSV LOAD
# -------------------------
pred_csv = os.path.join(PRED_ROOT, f"preds_{MODEL_NAME}_{TARGET_SPLIT}.csv")
assert os.path.exists(pred_csv), f"Missing prediction CSV: {pred_csv}"

pred_df = pd.read_csv(pred_csv).copy()
pred_df["clip_path"] = pred_df["path"].astype(str)
pred_df["y"] = pred_df["y_true"].astype(int)
pred_df["p1"] = pred_df["p_drowsy"].astype(float)
pred_df["yhat"] = (pred_df["p1"] >= thr).astype(int)

# -------------------------
# FIND TALKING FALSE POSITIVE
# -------------------------
fp = pred_df[(pred_df["y"] == 0) & (pred_df["yhat"] == 1)].copy()
fp = fp.sort_values("p1", ascending=False).reset_index(drop=True)

print(f"\nTotal false positives in {MODEL_NAME} / {TARGET_SPLIT}: {len(fp)}")

pattern = "|".join([k.replace("+", r"\+") for k in TALK_KEYWORDS])
talking_fp = fp[fp["clip_path"].str.lower().str.contains(pattern, regex=True, na=False)].copy()
talking_fp = talking_fp.sort_values("p1", ascending=False).reset_index(drop=True)

print(f"Likely talking false positives found by keyword: {len(talking_fp)}")

if len(talking_fp) > 0:
    chosen = talking_fp.iloc[0]
    chosen_reason = "keyword-matched talking false positive"
else:
    chosen = fp.iloc[0] if len(fp) > 0 else None
    chosen_reason = "fallback strongest false positive (no talking keyword match found)"

assert chosen is not None, "No false positives found."

clip_path = str(chosen["clip_path"])
p1 = float(chosen["p1"])
y = int(chosen["y"])

print("\nChosen clip:")
print("Reason :", chosen_reason)
print("Path   :", clip_path)
print("y_true :", y)
print("p1     :", p1)
print("thr    :", thr)

# -------------------------
# PATH RESOLUTION
# -------------------------
def resolve_clip_path(path: str) -> str:
    path = str(path).replace("\\", "/")
    if os.path.exists(path):
        return path

    local_prefix = CLIPS_LOCAL.rstrip("/") + "/"
    drive_prefix = CLIPS_DRIVE.rstrip("/") + "/"

    # local -> drive
    if path.startswith(local_prefix):
        rel = path[len(local_prefix):]
        candidate = drive_prefix + rel
        if os.path.exists(candidate):
            return candidate

    # drive -> local
    if path.startswith(drive_prefix):
        rel = path[len(drive_prefix):]
        candidate = local_prefix + rel
        if os.path.exists(candidate):
            return candidate

    # basename fallback under drive tree
    target_name = os.path.basename(path)
    matches = []
    for root, _, files in os.walk(CLIPS_DRIVE):
        if target_name in files:
            matches.append(os.path.join(root, target_name))
            if len(matches) >= 1:
                break
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not resolve clip path:\n{path}")

# -------------------------
# LOAD CLIP
# -------------------------
def load_npz_clip_direct(path, mode="cthw", clip_t=16, clip_h=112, clip_w=112, clip_c=3):
    resolved = resolve_clip_path(path)
    print("\nResolved clip path:")
    print(resolved)

    z = np.load(resolved, allow_pickle=False)
    x = z["x"] if "x" in z.files else z[z.files[0]]
    x = x.astype(np.float32)

    if x.ndim != 4:
        raise ValueError(f"bad ndim={x.ndim} for {resolved}")

    if x.shape[-1] in (1, 3):          # (T,H,W,C)
        x = np.transpose(x, (3, 0, 1, 2))
    elif x.shape[1] in (1, 3):         # (T,C,H,W)
        x = np.transpose(x, (1, 0, 2, 3))
    elif x.shape[0] in (1, 3):         # already (C,T,H,W)
        pass
    else:
        raise ValueError(f"unrecognized layout shape={x.shape} for {resolved}")

    expected = (clip_c, clip_t, clip_h, clip_w)
    if x.shape != expected:
        raise ValueError(f"wrong shape {x.shape}, expected {expected} for {resolved}")

    if not np.isfinite(x).all():
        raise ValueError(f"nan/inf found in {resolved}")

    if mode == "cthw":
        xt = torch.from_numpy(x)
    elif mode == "tchw":
        xt = torch.from_numpy(np.transpose(x, (1, 0, 2, 3)))
    else:
        raise ValueError("mode must be cthw or tchw")

    return xt, resolved

x, resolved_path = load_npz_clip_direct(clip_path, mode=mode)

# -------------------------
# VIS HELPERS
# -------------------------
def to_numpy_img(x_chw):
    x = x_chw.astype(np.float32)
    if x.max() > 1.5:
        x = x / 255.0
    x = np.clip(x, 0, 1)
    return np.transpose(x, (1, 2, 0))

def overlay_heatmap(img_hwc, cam_hw, alpha=0.45):
    cam = np.clip(cam_hw, 0, 1)
    heat = plt.get_cmap("jet")(cam)[:, :, :3]
    out = (1 - alpha) * img_hwc + alpha * heat
    return np.clip(out, 0, 1)

def save_grid(frames_hwc, cams_hw, title, save_path, max_frames=8):
    T = len(frames_hwc)
    if T <= max_frames:
        idxs = list(range(T))
    else:
        idxs = np.linspace(0, T - 1, max_frames).round().astype(int).tolist()

    n = len(idxs)
    cols = min(n, 4)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, t in enumerate(idxs):
        over = overlay_heatmap(frames_hwc[t], cams_hw[t])
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(over)
        ax.set_title(f"t={t}")
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def set_bn_eval(m):
    if isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
        m.eval()

def set_dropout_eval(m):
    if isinstance(m, torch.nn.Dropout):
        m.eval()

# -------------------------
# GRAD-CAM
# -------------------------
def gradcam_resnet18_video(model, x_cthw_or_tchw, mode, target_class=1, needs_rnn_train=False):
    conv = model.backbone[7][-1].conv2
    acts, grads = [], []

    def fwd_hook(m, inp, out):
        acts.append(out)

    def bwd_hook(m, gin, gout):
        grads.append(gout[0])

    h1 = conv.register_forward_hook(fwd_hook)
    h2 = conv.register_full_backward_hook(bwd_hook)

    was_training = model.training
    if needs_rnn_train:
        model.train()
        model.apply(set_bn_eval)
        model.apply(set_dropout_eval)
    else:
        model.eval()

    model.zero_grad(set_to_none=True)

    xb = x_cthw_or_tchw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    if was_training:
        model.train()
    else:
        model.eval()

    h1.remove()
    h2.remove()

    A = acts[0]
    G = grads[0]
    w = G.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * A).sum(dim=1))

    if mode == "cthw":
        T = xb.shape[2]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    else:
        T = xb.shape[1]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].detach().cpu().numpy()

    cam = cam.view(T, cam.shape[-2], cam.shape[-1]).unsqueeze(1)
    cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False).squeeze(1)
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

def gradcam_r3d18(model, x_cthw, target_class=1, layer="layer3"):
    acts, grads = [], []
    conv = model.layer3[1].conv2 if layer == "layer3" else model.layer4[1].conv2

    def fwd_hook(m, inp, out):
        acts.append(out)
        out.register_hook(lambda g: grads.append(g))

    h = conv.register_forward_hook(fwd_hook)

    model.eval()
    model.zero_grad(set_to_none=True)

    xb = x_cthw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    h.remove()

    A = acts[0]
    G = grads[0]
    w = G.mean(dim=(2, 3, 4), keepdim=True)
    cam = F.relu((w * A).sum(dim=1))

    T, H, W = xb.shape[2], xb.shape[3], xb.shape[4]
    cam = cam.unsqueeze(1)
    cam = F.interpolate(cam, size=(T, H, W), mode="trilinear", align_corners=False).squeeze(1)[0]
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

if MODEL_NAME == "r3d18":
    frames_hwc, cams = gradcam_r3d18(model, x, target_class=1, layer="layer3")
    title = f"{MODEL_NAME}(layer3) | {TARGET_SPLIT} | talking FP | y=0 p1={p1:.3f} thr={thr:.3f}"
    save_name = f"{MODEL_NAME}_layer3_{TARGET_SPLIT}_talking_fp.png"
else:
    needs_rnn = (MODEL_NAME == "cnn_gru")
    frames_hwc, cams = gradcam_resnet18_video(
        model, x, mode=mode, target_class=1, needs_rnn_train=needs_rnn
    )
    title = f"{MODEL_NAME} | {TARGET_SPLIT} | talking FP | y=0 p1={p1:.3f} thr={thr:.3f}"
    save_name = f"{MODEL_NAME}_{TARGET_SPLIT}_talking_fp.png"

save_path = os.path.join(OUT_DIR, save_name)
save_grid(frames_hwc, cams, title=title, save_path=save_path, max_frames=8)

meta = {
    "model": MODEL_NAME,
    "split": TARGET_SPLIT,
    "case": "talking_false_positive",
    "selection_reason": chosen_reason,
    "talk_keywords": TALK_KEYWORDS,
    "clip_path_from_csv": clip_path,
    "resolved_clip_path": resolved_path,
    "y_true": y,
    "p1": p1,
    "threshold": thr,
    "prediction_csv": pred_csv,
    "png": save_path,
}
with open(save_path.replace(".png", ".json"), "w") as f:
    json.dump(meta, f, indent=2)

print("\n✅ Done")
print("Saved image:", save_path)
print("Saved meta :", save_path.replace(".png", ".json"))

---
## 4. YawDD Yawning True-Positive

Selects a yawning clip correctly classified as drowsy (true positive) and visualises the model's attention, contrasting with the talking false-positive case above.

In [ ]:
# =========================================================
# YAWDD YAWNING TRUE POSITIVE + GRAD-CAM (ONE CELL)
# - finds a likely yawning true positive from saved predictions
# - runs Grad-CAM
# - saves PNG + JSON to Drive
# =========================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision

# -------------------------
# USER SETTINGS
# -------------------------
MODEL_NAME = "cnn_gru"   # "cnn_gru", "tsm_fast", or "r3d18"
TARGET_SPLIT = "YawDD_test"

YAWN_KEYWORDS = [
    "yawn", "yawning"
]

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
PRED_ROOT = os.path.join(SAVE_ROOT, "final_test_eval")
OUT_DIR   = os.path.join(SAVE_ROOT, MODEL_NAME, "gradcam_yawning_tp", TARGET_SPLIT)

CLIPS_DRIVE = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL = "/content/Clips"

THR = {
    "tsm_fast": 0.010,
    "cnn_gru":  0.001,
    "r3d18":    0.004
}

# -------------------------
# SETUP
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 2
os.makedirs(OUT_DIR, exist_ok=True)

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

thr = THR[MODEL_NAME]

# -------------------------
# MODEL BUILDERS
# -------------------------
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous().view(B * T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, dropout=0.30, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        out_last = self.drop(out[:, -1, :])
        return self.fc(out_last)

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, dropout=0.30, num_classes=num_classes)

def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

BUILDERS = {
    "tsm_fast": build_tsm_fast,
    "cnn_gru": build_cnn_gru,
    "r3d18": build_r3d18
}

MODES = {
    "tsm_fast": "cthw",
    "cnn_gru": "tchw",
    "r3d18": "cthw"
}

def load_ckpt(builder, ckpt_path):
    model = builder(NUM_CLASSES).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()
    return model

ckpt_path = os.path.join(SAVE_ROOT, MODEL_NAME, "best.pt")
assert os.path.exists(ckpt_path), f"Missing checkpoint: {ckpt_path}"
model = load_ckpt(BUILDERS[MODEL_NAME], ckpt_path)
mode = MODES[MODEL_NAME]

# -------------------------
# PREDICTION CSV LOAD
# -------------------------
pred_csv = os.path.join(PRED_ROOT, f"preds_{MODEL_NAME}_{TARGET_SPLIT}.csv")
assert os.path.exists(pred_csv), f"Missing prediction CSV: {pred_csv}"

pred_df = pd.read_csv(pred_csv).copy()
pred_df["clip_path"] = pred_df["path"].astype(str)
pred_df["y"] = pred_df["y_true"].astype(int)
pred_df["p1"] = pred_df["p_drowsy"].astype(float)
pred_df["yhat"] = (pred_df["p1"] >= thr).astype(int)

# -------------------------
# FIND YAWNING TRUE POSITIVE
# -------------------------
tp = pred_df[(pred_df["y"] == 1) & (pred_df["yhat"] == 1)].copy()
tp = tp.sort_values("p1", ascending=False).reset_index(drop=True)

print(f"\nTotal true positives in {MODEL_NAME} / {TARGET_SPLIT}: {len(tp)}")

pattern = "|".join([k.replace("+", r"\+") for k in YAWN_KEYWORDS])
yawn_tp = tp[tp["clip_path"].str.lower().str.contains(pattern, regex=True, na=False)].copy()
yawn_tp = yawn_tp.sort_values("p1", ascending=False).reset_index(drop=True)

print(f"Likely yawning true positives found by keyword: {len(yawn_tp)}")

if len(yawn_tp) > 0:
    chosen = yawn_tp.iloc[0]
    chosen_reason = "keyword-matched yawning true positive"
else:
    chosen = tp.iloc[0] if len(tp) > 0 else None
    chosen_reason = "fallback strongest true positive (no yawn keyword match found)"

assert chosen is not None, "No true positives found."

clip_path = str(chosen["clip_path"])
p1 = float(chosen["p1"])
y = int(chosen["y"])

print("\nChosen clip:")
print("Reason :", chosen_reason)
print("Path   :", clip_path)
print("y_true :", y)
print("p1     :", p1)
print("thr    :", thr)

# -------------------------
# PATH RESOLUTION
# -------------------------
def resolve_clip_path(path: str) -> str:
    path = str(path).replace("\\", "/")
    if os.path.exists(path):
        return path

    local_prefix = CLIPS_LOCAL.rstrip("/") + "/"
    drive_prefix = CLIPS_DRIVE.rstrip("/") + "/"

    if path.startswith(local_prefix):
        rel = path[len(local_prefix):]
        candidate = drive_prefix + rel
        if os.path.exists(candidate):
            return candidate

    if path.startswith(drive_prefix):
        rel = path[len(drive_prefix):]
        candidate = local_prefix + rel
        if os.path.exists(candidate):
            return candidate

    target_name = os.path.basename(path)
    for root, _, files in os.walk(CLIPS_DRIVE):
        if target_name in files:
            return os.path.join(root, target_name)

    raise FileNotFoundError(f"Could not resolve clip path:\n{path}")

# -------------------------
# LOAD CLIP
# -------------------------
def load_npz_clip_direct(path, mode="cthw", clip_t=16, clip_h=112, clip_w=112, clip_c=3):
    resolved = resolve_clip_path(path)
    print("\nResolved clip path:")
    print(resolved)

    z = np.load(resolved, allow_pickle=False)
    x = z["x"] if "x" in z.files else z[z.files[0]]
    x = x.astype(np.float32)

    if x.ndim != 4:
        raise ValueError(f"bad ndim={x.ndim} for {resolved}")

    if x.shape[-1] in (1, 3):          # (T,H,W,C)
        x = np.transpose(x, (3, 0, 1, 2))
    elif x.shape[1] in (1, 3):         # (T,C,H,W)
        x = np.transpose(x, (1, 0, 2, 3))
    elif x.shape[0] in (1, 3):         # already (C,T,H,W)
        pass
    else:
        raise ValueError(f"unrecognized layout shape={x.shape} for {resolved}")

    expected = (clip_c, clip_t, clip_h, clip_w)
    if x.shape != expected:
        raise ValueError(f"wrong shape {x.shape}, expected {expected} for {resolved}")

    if not np.isfinite(x).all():
        raise ValueError(f"nan/inf found in {resolved}")

    if mode == "cthw":
        xt = torch.from_numpy(x)
    elif mode == "tchw":
        xt = torch.from_numpy(np.transpose(x, (1, 0, 2, 3)))
    else:
        raise ValueError("mode must be cthw or tchw")

    return xt, resolved

x, resolved_path = load_npz_clip_direct(clip_path, mode=mode)

# -------------------------
# VIS HELPERS
# -------------------------
def to_numpy_img(x_chw):
    x = x_chw.astype(np.float32)
    if x.max() > 1.5:
        x = x / 255.0
    x = np.clip(x, 0, 1)
    return np.transpose(x, (1, 2, 0))

def overlay_heatmap(img_hwc, cam_hw, alpha=0.45):
    cam = np.clip(cam_hw, 0, 1)
    heat = plt.get_cmap("jet")(cam)[:, :, :3]
    out = (1 - alpha) * img_hwc + alpha * heat
    return np.clip(out, 0, 1)

def save_grid(frames_hwc, cams_hw, title, save_path, max_frames=8):
    T = len(frames_hwc)
    if T <= max_frames:
        idxs = list(range(T))
    else:
        idxs = np.linspace(0, T - 1, max_frames).round().astype(int).tolist()

    n = len(idxs)
    cols = min(n, 4)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, t in enumerate(idxs):
        over = overlay_heatmap(frames_hwc[t], cams_hw[t])
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(over)
        ax.set_title(f"t={t}")
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def set_bn_eval(m):
    if isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
        m.eval()

def set_dropout_eval(m):
    if isinstance(m, torch.nn.Dropout):
        m.eval()

# -------------------------
# GRAD-CAM
# -------------------------
def gradcam_resnet18_video(model, x_cthw_or_tchw, mode, target_class=1, needs_rnn_train=False):
    conv = model.backbone[7][-1].conv2
    acts, grads = [], []

    def fwd_hook(m, inp, out):
        acts.append(out)

    def bwd_hook(m, gin, gout):
        grads.append(gout[0])

    h1 = conv.register_forward_hook(fwd_hook)
    h2 = conv.register_full_backward_hook(bwd_hook)

    was_training = model.training
    if needs_rnn_train:
        model.train()
        model.apply(set_bn_eval)
        model.apply(set_dropout_eval)
    else:
        model.eval()

    model.zero_grad(set_to_none=True)

    xb = x_cthw_or_tchw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    if was_training:
        model.train()
    else:
        model.eval()

    h1.remove()
    h2.remove()

    A = acts[0]
    G = grads[0]
    w = G.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * A).sum(dim=1))

    if mode == "cthw":
        T = xb.shape[2]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    else:
        T = xb.shape[1]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].detach().cpu().numpy()

    cam = cam.view(T, cam.shape[-2], cam.shape[-1]).unsqueeze(1)
    cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False).squeeze(1)
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

def gradcam_r3d18(model, x_cthw, target_class=1, layer="layer3"):
    acts, grads = [], []
    conv = model.layer3[1].conv2 if layer == "layer3" else model.layer4[1].conv2

    def fwd_hook(m, inp, out):
        acts.append(out)
        out.register_hook(lambda g: grads.append(g))

    h = conv.register_forward_hook(fwd_hook)

    model.eval()
    model.zero_grad(set_to_none=True)

    xb = x_cthw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    h.remove()

    A = acts[0]
    G = grads[0]
    w = G.mean(dim=(2, 3, 4), keepdim=True)
    cam = F.relu((w * A).sum(dim=1))

    T, H, W = xb.shape[2], xb.shape[3], xb.shape[4]
    cam = cam.unsqueeze(1)
    cam = F.interpolate(cam, size=(T, H, W), mode="trilinear", align_corners=False).squeeze(1)[0]
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()
    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

if MODEL_NAME == "r3d18":
    frames_hwc, cams = gradcam_r3d18(model, x, target_class=1, layer="layer3")
    title = f"{MODEL_NAME}(layer3) | {TARGET_SPLIT} | yawning TP | y=1 p1={p1:.3f} thr={thr:.3f}"
    save_name = f"{MODEL_NAME}_layer3_{TARGET_SPLIT}_yawning_tp.png"
else:
    needs_rnn = (MODEL_NAME == "cnn_gru")
    frames_hwc, cams = gradcam_resnet18_video(
        model, x, mode=mode, target_class=1, needs_rnn_train=needs_rnn
    )
    title = f"{MODEL_NAME} | {TARGET_SPLIT} | yawning TP | y=1 p1={p1:.3f} thr={thr:.3f}"
    save_name = f"{MODEL_NAME}_{TARGET_SPLIT}_yawning_tp.png"

save_path = os.path.join(OUT_DIR, save_name)
save_grid(frames_hwc, cams, title=title, save_path=save_path, max_frames=8)

meta = {
    "model": MODEL_NAME,
    "split": TARGET_SPLIT,
    "case": "yawning_true_positive",
    "selection_reason": chosen_reason,
    "yawn_keywords": YAWN_KEYWORDS,
    "clip_path_from_csv": clip_path,
    "resolved_clip_path": resolved_path,
    "y_true": y,
    "p1": p1,
    "threshold": thr,
    "prediction_csv": pred_csv,
    "png": save_path,
}
with open(save_path.replace(".png", ".json"), "w") as f:
    json.dump(meta, f, indent=2)

print("\n✅ Done")
print("Saved image:", save_path)
print("Saved meta :", save_path.replace(".png", ".json"))